In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../data/processed/bangalore_traffic_weather.csv",
                 parse_dates=["LocalDateTime"]).set_index("LocalDateTime")

# Drop text columns that carry no predictive information
df = df.drop(columns=["Country", "City", "UpdateTimeUTC", "LocalDateTimeWeekAgo"])
df = df.asfreq("h")

# rain and precipitation looked identical in the summary table; check, then drop one
print("Largest difference between rain and precipitation:", (df["rain"] - df["precipitation"]).abs().max())
if np.allclose(df["rain"], df["precipitation"]):
    df = df.drop(columns=["rain"])
    print("Identical, so rain was dropped")

print("\nShape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("\nColumns:")
print(df.columns.tolist())

Largest difference between rain and precipitation: 0.0
Identical, so rain was dropped

Shape: (15829, 15)
Missing values: 0

Columns:
['JamsDelay', 'TrafficIndexLive', 'JamsLengthInKms', 'JamsCount', 'TrafficIndexWeekAgo', 'TravelTimeLivePer10KmsMins', 'TravelTimeHistoricPer10KmsMins', 'MinsDelay', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'cloud_cover', 'wind_speed_10m', 'is_raining', 'rain_3h']


In [2]:
cal = pd.DataFrame(index=df.index)

cal["hour"] = df.index.hour
cal["day_of_week"] = df.index.day_of_week          # Monday = 0 ... Sunday = 6
cal["is_sunday"] = (cal["day_of_week"] == 6).astype(int)

# Optional: wet-season flag (May to Oct, based on the rainfall table we saw)
cal["is_monsoon"] = df.index.month.isin([5, 6, 7, 8, 9, 10]).astype(int)

# Cyclical encodings (mainly for the LSTM)
cal["hour_sin"] = np.sin(2 * np.pi * cal["hour"] / 24)
cal["hour_cos"] = np.cos(2 * np.pi * cal["hour"] / 24)
cal["dow_sin"] = np.sin(2 * np.pi * cal["day_of_week"] / 7)
cal["dow_cos"] = np.cos(2 * np.pi * cal["day_of_week"] / 7)

print("Shape:", cal.shape)
print("Missing values:", cal.isna().sum().sum())
print("\nRows per day of week (0 = Monday):")
print(cal["day_of_week"].value_counts().sort_index().to_string())
print("\nSunday share:", round(cal["is_sunday"].mean(), 3), "(expect about 0.143)")
print("Monsoon share:", round(cal["is_monsoon"].mean(), 3))
print("\nSine/cosine ranges (should be between -1 and 1):")
print(cal[["hour_sin", "hour_cos", "dow_sin", "dow_cos"]].agg(["min", "max"]).round(2).to_string())
print("\nSample rows:")
display(cal.iloc[[0, 8, 24, 100]])

Shape: (15829, 8)
Missing values: 0

Rows per day of week (0 = Monday):
day_of_week
0    2256
1    2269
2    2280
3    2256
4    2256
5    2256
6    2256

Sunday share: 0.143 (expect about 0.143)
Monsoon share: 0.553

Sine/cosine ranges (should be between -1 and 1):
     hour_sin  hour_cos  dow_sin  dow_cos
min      -1.0      -1.0    -0.97     -0.9
max       1.0       1.0     0.97      1.0

Sample rows:


,hour,day_of_week,is_sunday,is_monsoon,hour_sin,hour_cos,dow_sin,dow_cos
LocalDateTime,,,,,,,,
2025-01-07 11:00:00,11,1,0,0,0.258819,-0.965926,0.781831,0.623490
2025-01-07 19:00:00,19,1,0,0,-0.965926,0.258819,0.781831,0.623490
2025-01-08 11:00:00,11,2,0,0,0.258819,-0.965926,0.974928,-0.222521
2025-01-11 15:00:00,15,5,0,0,-0.707107,-0.707107,-0.974928,-0.222521


In [3]:
# Karnataka general holidays from the calendar you provided
holiday_list = [
    # 2025
    ("2025-01-14", "Makara Sankranti"), ("2025-01-26", "Republic Day"),
    ("2025-02-26", "Maha Shivaratri"), ("2025-03-30", "Ugadi"),
    ("2025-03-31", "Eid-ul-Fitr"), ("2025-04-10", "Mahaveera Jayanthi"),
    ("2025-04-14", "Ambedkar Jayanthi"), ("2025-04-18", "Good Friday"),
    ("2025-04-30", "Basava Jayanthi"), ("2025-05-01", "May Day"),
    ("2025-06-07", "Bakrid"), ("2025-07-06", "Last Day of Moharram"),
    ("2025-08-15", "Independence Day"), ("2025-08-27", "Ganesh Chaturthi"),
    ("2025-09-05", "Eid-Milad"), ("2025-10-01", "Mahanavami"),
    ("2025-10-02", "Vijayadasami / Gandhi Jayanthi"), ("2025-10-07", "Valmiki Jayanthi"),
    ("2025-10-20", "Naraka Chaturdashi"), ("2025-10-22", "Balipadyami / Deepavali"),
    ("2025-11-01", "Kannada Rajyothsava"), ("2025-11-08", "Kanakadasa Jayanthi"),
    ("2025-12-25", "Christmas"),
    # 2026
    ("2026-01-15", "Makara Sankranti"), ("2026-01-26", "Republic Day"),
    ("2026-03-02", "Holi"), ("2026-03-19", "Ugadi"),
    ("2026-03-27", "Sri Ramanavami"), ("2026-04-03", "Good Friday"),
    ("2026-04-14", "Ambedkar Jayanthi"), ("2026-05-01", "May Day"),
    ("2026-05-27", "Bakrid"), ("2026-06-26", "Last Day of Muharram"),
    ("2026-08-15", "Independence Day"), ("2026-08-26", "Eid-e-Milad"),
    ("2026-09-14", "Ganesh Chaturthi"), ("2026-10-02", "Gandhi Jayanti"),
    ("2026-10-20", "Mahanavami / Ayudha Pooja"), ("2026-10-21", "Vijayadashami"),
    ("2026-11-10", "Balipadyami / Deepavali"), ("2026-11-27", "Kanakadasa Jayanti"),
    ("2026-12-25", "Christmas"),
]

hol = pd.DataFrame(holiday_list, columns=["date", "name"])
hol["date"] = pd.to_datetime(hol["date"])

# Keep only holidays inside the data period
hol = hol[(hol["date"] >= df.index.min().normalize()) & (hol["date"] <= df.index.max().normalize())]
hol = hol.sort_values("date").reset_index(drop=True)
hol["weekday"] = hol["date"].dt.day_name()

print("Holidays in the data period:", len(hol))
print("Of these, on a Sunday:", int((hol["weekday"] == "Sunday").sum()))
print(hol.to_string(index=False))

# Flag every hour that falls on a holiday
cal["is_holiday"] = df.index.normalize().isin(hol["date"]).astype(int)
print("\nHoliday hours:", cal["is_holiday"].sum(), "| holiday days:", cal["is_holiday"].sum() // 24)

Holidays in the data period: 39
Of these, on a Sunday: 3
      date                           name   weekday
2025-01-14               Makara Sankranti   Tuesday
2025-01-26                   Republic Day    Sunday
2025-02-26                Maha Shivaratri Wednesday
2025-03-30                          Ugadi    Sunday
2025-03-31                    Eid-ul-Fitr    Monday
2025-04-10             Mahaveera Jayanthi  Thursday
2025-04-14              Ambedkar Jayanthi    Monday
2025-04-18                    Good Friday    Friday
2025-04-30                Basava Jayanthi Wednesday
2025-05-01                        May Day  Thursday
2025-06-07                         Bakrid  Saturday
2025-07-06           Last Day of Moharram    Sunday
2025-08-15               Independence Day    Friday
2025-08-27               Ganesh Chaturthi Wednesday
2025-09-05                      Eid-Milad    Friday
2025-10-01                     Mahanavami Wednesday
2025-10-02 Vijayadasami / Gandhi Jayanthi  Thursday
2025-10

In [4]:
y = df["TrafficIndexLive"]
d = pd.DataFrame({"y": y, "hour": df.index.hour,
                  "dow": df.index.day_of_week, "hol": cal["is_holiday"]})

d["group"] = np.where(d["dow"] == 6, "Sunday",
              np.where(d["hol"] == 1, "Holiday (Mon-Sat)", "Ordinary Mon-Sat"))

day = d[(d["hour"] >= 8) & (d["hour"] <= 21)]
print("Daytime 8am-9pm:")
print(day.groupby("group")["y"].agg(["mean", "count"]).round(1).to_string())

eve = d[d["hour"].isin([18, 19, 20, 21])]
print("\nEvening 6-9 pm:")
print(eve.groupby("group")["y"].agg(["mean", "count"]).round(1).to_string())

Daytime 8am-9pm:
                   mean  count
group                         
Holiday (Mon-Sat)  36.7    504
Ordinary Mon-Sat   52.8   7417
Sunday             31.6   1316

Evening 6-9 pm:
                   mean  count
group                         
Holiday (Mon-Sat)  49.6    144
Ordinary Mon-Sat   66.7   2120
Sunday             40.8    376


In [5]:
y = df["TrafficIndexLive"]
lagf = pd.DataFrame(index=df.index)

# Value k hours before the forecast origin (lag 0 = the current hour, known at the origin)
for k in [0, 1, 2, 3, 6, 12, 24, 168]:
    lagf[f"y_lag{k}"] = y.shift(k)

# Rolling summaries over the hours up to and including the origin
lagf["y_mean24"] = y.rolling(24).mean()
lagf["y_mean168"] = y.rolling(168).mean()
lagf["y_std24"] = y.rolling(24).std()

print("Shape:", lagf.shape)
print("\nMissing values per column (these rows are dropped in step 3.9):")
print(lagf.isna().sum().to_string())

# Spot-check one timestamp by hand
t = pd.Timestamp("2026-10-20 19:00")
print("\nSpot check at", t)
print("lag0   :", y.loc[t], "|", lagf.loc[t, "y_lag0"])
print("lag24  :", y.loc[t - pd.Timedelta(hours=24)], "|", lagf.loc[t, "y_lag24"])
print("lag168 :", y.loc[t - pd.Timedelta(hours=168)], "|", lagf.loc[t, "y_lag168"])
print("mean24 :", round(y.loc[t - pd.Timedelta(hours=23):t].mean(), 3), "|", round(lagf.loc[t, "y_mean24"], 3))

# How strongly does each feature relate to the value 1 and 6 hours ahead?
print("\nCorrelation with the value 1h and 6h ahead:")
out = pd.DataFrame({
    "1h_ahead": lagf.corrwith(y.shift(-1)),
    "6h_ahead": lagf.corrwith(y.shift(-6)),
}).round(3)
print(out.to_string())

Shape: (15829, 11)

Missing values per column (these rows are dropped in step 3.9):
y_lag0         0
y_lag1         1
y_lag2         2
y_lag3         3
y_lag6         6
y_lag12       12
y_lag24       24
y_lag168     168
y_mean24      23
y_mean168    167
y_std24       23

Spot check at 2026-10-20 19:00:00
lag0   : 41.0 | 41.0
lag24  : 65.0 | 65.0
lag168 : 75.0 | 75.0
mean24 : 18.333 | 18.333

Correlation with the value 1h and 6h ahead:
           1h_ahead  6h_ahead
y_lag0        0.935    -0.005
y_lag1        0.776    -0.156
y_lag2        0.570    -0.305
y_lag3        0.356    -0.452
y_lag6       -0.156    -0.735
y_lag12      -0.706    -0.050
y_lag24       0.848    -0.038
y_lag168      0.878    -0.030
y_mean24      0.148     0.095
y_mean168     0.090     0.080
y_std24       0.157     0.082


In [6]:
jam_cols = ["JamsCount", "JamsLengthInKms", "JamsDelay", "MinsDelay", "TravelTimeLivePer10KmsMins"]
short = {"JamsCount": "jcount", "JamsLengthInKms": "jlen", "JamsDelay": "jdelay",
         "MinsDelay": "mdelay", "TravelTimeLivePer10KmsMins": "tt"}

jamf = pd.DataFrame(index=df.index)
for c in jam_cols:
    for k in [0, 1, 3]:
        jamf[f"{short[c]}_lag{k}"] = df[c].shift(k)

print("Shape:", jamf.shape)
print("Missing values (max per column):", int(jamf.isna().sum().max()))

# Spot check
t = pd.Timestamp("2026-10-20 19:00")
print("\nSpot check at", t)
print("JamsCount lag0:", df.loc[t, "JamsCount"], "|", jamf.loc[t, "jcount_lag0"])
print("JamsCount lag3:", df.loc[t - pd.Timedelta(hours=3), "JamsCount"], "|", jamf.loc[t, "jcount_lag3"])

# Does it add information beyond the target's own lag0?
y = df["TrafficIndexLive"]
print("\nCorrelation with the value 1h and 6h ahead:")
print(pd.DataFrame({"1h_ahead": jamf.corrwith(y.shift(-1)),
                    "6h_ahead": jamf.corrwith(y.shift(-6))}).round(3).to_string())

print("\nCorrelation of jam lag0 columns with y_lag0 (how redundant they are):")
print(jamf[[c for c in jamf.columns if c.endswith("lag0")]].corrwith(y).round(3).to_string())

Shape: (15829, 15)
Missing values (max per column): 3

Spot check at 2026-10-20 19:00:00
JamsCount lag0: 66.0 | 66.0
JamsCount lag3: 28.0 | 28.0

Correlation with the value 1h and 6h ahead:
             1h_ahead  6h_ahead
jcount_lag0     0.911     0.007
jcount_lag1     0.776    -0.136
jcount_lag3     0.368    -0.418
jlen_lag0       0.905    -0.011
jlen_lag1       0.766    -0.141
jlen_lag3       0.346    -0.393
jdelay_lag0     0.870    -0.033
jdelay_lag1     0.729    -0.165
jdelay_lag3     0.319    -0.418
mdelay_lag0     0.434    -0.110
mdelay_lag1     0.301    -0.146
mdelay_lag3     0.043    -0.186
tt_lag0         0.925    -0.011
tt_lag1         0.784    -0.171
tt_lag3         0.374    -0.452

Correlation of jam lag0 columns with y_lag0 (how redundant they are):
jcount_lag0    0.943
jlen_lag0      0.935
jdelay_lag0    0.909
mdelay_lag0    0.536
tt_lag0        0.980


In [7]:
wx = pd.DataFrame(index=df.index)

for c in ["temperature_2m", "relative_humidity_2m", "cloud_cover", "wind_speed_10m", "precipitation"]:
    wx[c] = df[c]
wx["is_raining"] = df["is_raining"]

# Recent rainfall totals up to and including the origin hour
for w in [3, 6, 12, 24]:
    wx[f"rain_{w}h"] = df["precipitation"].rolling(w).sum()
wx["precip_lag1"] = df["precipitation"].shift(1)

# Sudden changes in temperature (possible storm signal)
wx["temp_change_1h"] = df["temperature_2m"].diff(1)
wx["temp_change_3h"] = df["temperature_2m"].diff(3)

print("Shape:", wx.shape)
print("\nMissing values per column:")
print(wx.isna().sum()[lambda s: s > 0].to_string())

# Check rain_3h matches the one we made earlier (after the first 2 rows)
print("\nrain_3h matches earlier column:", np.allclose(wx["rain_3h"].iloc[2:], df["rain_3h"].iloc[2:]))

# Spot check on the 29 April 2026 event
t = pd.Timestamp("2026-04-29 17:00")
print("\nTemperature at 16:00 and 17:00:", df.loc[t - pd.Timedelta(hours=1), "temperature_2m"], df.loc[t, "temperature_2m"])
print("temp_change_1h at 17:00:", round(wx.loc[t, "temp_change_1h"], 1))

# How often does the temperature fall sharply in one hour?
drops = wx["temp_change_1h"] <= -3
print("\nHours with a temperature drop of 3 C or more:", int(drops.sum()))
print(pd.Series(wx.index[drops].strftime("%Y-%m-%d %H:%M")).head(15).to_string(index=False))

y = df["TrafficIndexLive"]
print("\nCorrelation with the value 1h and 6h ahead:")
print(pd.DataFrame({"1h_ahead": wx.corrwith(y.shift(-1)),
                    "6h_ahead": wx.corrwith(y.shift(-6))}).round(3).to_string())

Shape: (15829, 13)

Missing values per column:
rain_3h            2
rain_6h            5
rain_12h          11
rain_24h          23
precip_lag1        1
temp_change_1h     1
temp_change_3h     3

rain_3h matches earlier column: True

Temperature at 16:00 and 17:00: 35.1 30.2
temp_change_1h at 17:00: -4.9

Hours with a temperature drop of 3 C or more: 108
2025-03-11 19:00
2025-03-12 16:00
2025-03-17 19:00
2025-03-22 15:00
2025-04-03 15:00
2025-04-04 18:00
2025-04-11 17:00
2025-04-11 19:00
2025-04-12 17:00
2025-04-13 18:00
2025-04-14 17:00
2025-04-16 15:00
2025-04-16 16:00
2025-04-18 16:00
2025-04-20 18:00

Correlation with the value 1h and 6h ahead:
                      1h_ahead  6h_ahead
temperature_2m           0.574     0.229
relative_humidity_2m    -0.482    -0.140
cloud_cover              0.032     0.081
wind_speed_10m           0.146     0.129
precipitation            0.131     0.017
is_raining               0.116     0.003
rain_3h                  0.177    -0.036
rain_6h         

In [8]:
# Combine all origin-level feature tables
X = pd.concat([cal, lagf, jamf, wx], axis=1)

# Safety check: the raw target must never be a feature, and no duplicated names
assert "TrafficIndexLive" not in X.columns
assert not X.columns.duplicated().any()

groups = {
    "calendar": ["hour", "day_of_week", "is_sunday", "is_holiday",
                 "hour_sin", "hour_cos", "dow_sin", "dow_cos"],
    "monsoon_flag": ["is_monsoon"],
    "target_history": list(lagf.columns),
    "jam": list(jamf.columns),
    "weather_basic": ["temperature_2m", "relative_humidity_2m", "cloud_cover",
                      "wind_speed_10m", "precipitation", "is_raining",
                      "rain_3h", "rain_6h", "rain_12h", "rain_24h", "precip_lag1"],
    "weather_tempchange": ["temp_change_1h", "temp_change_3h"],
}

# Every column must belong to exactly one group
all_assigned = [c for g in groups.values() for c in g]
print("Total columns in X:", X.shape[1])
print("Assigned to a group:", len(all_assigned))
print("Unassigned columns:", sorted(set(X.columns) - set(all_assigned)))
print("Names in groups but not in X:", sorted(set(all_assigned) - set(X.columns)))
print("Assigned twice:", sorted({c for c in all_assigned if all_assigned.count(c) > 1}))

print("\nGroup sizes:")
for name, cols in groups.items():
    print(f"  {name:<20} {len(cols)}")

print("\nMax missing values in any column:", int(X.isna().sum().max()))
print("Rows with at least one missing value:", int(X.isna().any(axis=1).sum()))

Total columns in X: 48
Assigned to a group: 48
Unassigned columns: []
Names in groups but not in X: []
Assigned twice: []

Group sizes:
  calendar             8
  monsoon_flag         1
  target_history       11
  jam                  15
  weather_basic        11
  weather_tempchange   2

Max missing values in any column: 168
Rows with at least one missing value: 168


In [9]:
H = [1, 3, 6]
y = df["TrafficIndexLive"]
data, groups_h = {}, {}

for h in H:
    assert 168 - h > 0 and 24 - h > 0    # same-hour features must look backward from the origin

    tgt_time = df.index + pd.Timedelta(hours=h)      # the hour being predicted
    T = pd.DataFrame(index=df.index)

    # Calendar of the target hour
    T["t_hour"] = tgt_time.hour
    T["t_dow"] = tgt_time.day_of_week
    T["t_is_sunday"] = (T["t_dow"] == 6).astype(int)
    T["t_is_holiday"] = tgt_time.normalize().isin(hol["date"]).astype(int)
    T["t_hour_sin"] = np.sin(2 * np.pi * T["t_hour"] / 24)
    T["t_hour_cos"] = np.cos(2 * np.pi * T["t_hour"] / 24)
    T["t_dow_sin"] = np.sin(2 * np.pi * T["t_dow"] / 7)
    T["t_dow_cos"] = np.cos(2 * np.pi * T["t_dow"] / 7)

    # Same hour yesterday / last week, aligned to the target hour (known at the origin)
    T["same_hour_yday"] = y.shift(24 - h)
    T["same_hour_lastweek"] = y.shift(168 - h)

    # TomTom benchmark columns at the target hour (assumed known in advance, unverified)
    T["tt_weekago_idx"] = df["TrafficIndexWeekAgo"].shift(-h)
    T["tt_hist_traveltime"] = df["TravelTimeHistoricPer10KmsMins"].shift(-h)

    # Weather at the target hour (upper-bound experiment only)
    T["w_precip"] = df["precipitation"].shift(-h)
    T["w_is_raining"] = df["is_raining"].shift(-h)
    T["w_temp"] = df["temperature_2m"].shift(-h)

    # The target
    T["target"] = y.shift(-h)

    data[h] = pd.concat([X, T], axis=1)
    groups_h[h] = {
        **groups,
        "calendar_target": ["t_hour", "t_dow", "t_is_sunday", "t_is_holiday",
                            "t_hour_sin", "t_hour_cos", "t_dow_sin", "t_dow_cos"],
        "history_target_hour": ["same_hour_yday", "same_hour_lastweek"],
        "tomtom_benchmark": ["tt_weekago_idx", "tt_hist_traveltime"],
        "weather_target_hour_UPPER_BOUND": ["w_precip", "w_is_raining", "w_temp"],
    }

    print(f"h = {h}:  shape {data[h].shape}")
    print("  missing in new columns:", T.isna().sum()[lambda s: s > 0].to_dict())

# Spot check: origin 2026-10-20 19:00, horizon 3 -> predicting 22:00
t, h = pd.Timestamp("2026-10-20 19:00"), 3
row = data[h].loc[t]
print("\nSpot check, origin", t, "| predicting", t + pd.Timedelta(hours=h))
print("target             :", row["target"], "| actual:", y.loc[t + pd.Timedelta(hours=h)])
print("same_hour_yday     :", row["same_hour_yday"], "| actual:", y.loc[t + pd.Timedelta(hours=h) - pd.Timedelta(hours=24)])
print("same_hour_lastweek :", row["same_hour_lastweek"], "| actual:", y.loc[t + pd.Timedelta(hours=h) - pd.Timedelta(hours=168)])
print("t_hour, t_dow, t_is_holiday:", row["t_hour"], row["t_dow"], row["t_is_holiday"])

# Sanity check on the whole data (not the final baseline, which uses the test period only)
print("\nMAE of simple copy-forward rules over all available rows:")
for h in H:
    d = data[h].dropna(subset=["target", "same_hour_yday", "same_hour_lastweek"])
    print(f"  h={h}:  same hour yesterday {np.abs(d['target'] - d['same_hour_yday']).mean():.2f}"
          f"  |  same hour last week {np.abs(d['target'] - d['same_hour_lastweek']).mean():.2f}")

h = 1:  shape (15829, 64)
  missing in new columns: {'same_hour_yday': 23, 'same_hour_lastweek': 167, 'tt_weekago_idx': 1, 'tt_hist_traveltime': 1, 'w_precip': 1, 'w_is_raining': 1, 'w_temp': 1, 'target': 1}
h = 3:  shape (15829, 64)
  missing in new columns: {'same_hour_yday': 21, 'same_hour_lastweek': 165, 'tt_weekago_idx': 3, 'tt_hist_traveltime': 3, 'w_precip': 3, 'w_is_raining': 3, 'w_temp': 3, 'target': 3}
h = 6:  shape (15829, 64)
  missing in new columns: {'same_hour_yday': 18, 'same_hour_lastweek': 162, 'tt_weekago_idx': 6, 'tt_hist_traveltime': 6, 'w_precip': 6, 'w_is_raining': 6, 'w_temp': 6, 'target': 6}

Spot check, origin 2026-10-20 19:00:00 | predicting 2026-10-20 22:00:00
target             : 22.0 | actual: 22.0
same_hour_yday     : 40.0 | actual: 40.0
same_hour_lastweek : 26.0 | actual: 26.0
t_hour, t_dow, t_is_holiday: 22.0 1.0 1.0

MAE of simple copy-forward rules over all available rows:
  h=1:  same hour yesterday 6.93  |  same hour last week 4.96
  h=3:  same hour

In [10]:
clean = {}
for h in H:
    d = data[h]
    before = len(d)
    d = d.dropna()
    clean[h] = d
    print(f"h = {h}:  {before} -> {len(d)} rows  (dropped {before - len(d)})")
    print(f"   first origin: {d.index.min()}   last origin: {d.index.max()}")

    # Checks
    assert d.isna().sum().sum() == 0
    assert d.index.is_monotonic_increasing
    steps = d.index.to_series().diff().dropna().unique()
    assert len(steps) == 1 and steps[0] == pd.Timedelta(hours=1)   # still one continuous hourly block
    assert "TrafficIndexLive" not in d.columns

print("\nAll checks passed")
print("Columns per horizon:", clean[1].shape[1])

h = 1:  15829 -> 15660 rows  (dropped 169)
   first origin: 2025-01-14 11:00:00   last origin: 2026-10-28 22:00:00
h = 3:  15829 -> 15658 rows  (dropped 171)
   first origin: 2025-01-14 11:00:00   last origin: 2026-10-28 20:00:00
h = 6:  15829 -> 15655 rows  (dropped 174)
   first origin: 2025-01-14 11:00:00   last origin: 2026-10-28 17:00:00

All checks passed
Columns per horizon: 64


In [11]:
T_END      = pd.Timestamp("2026-06-30 23:00")
V_START    = pd.Timestamp("2026-07-01 00:00")
V_END      = pd.Timestamp("2026-08-31 23:00")
TEST_START = pd.Timestamp("2026-09-01 00:00")

splits = {}
for h in H:
    d = clean[h]
    tgt_time = d.index + pd.Timedelta(hours=h)

    train = d[tgt_time <= T_END]
    val   = d[(d.index >= V_START) & (tgt_time <= V_END)]
    test  = d[d.index >= TEST_START]
    splits[h] = {"train": train, "val": val, "test": test}

    # Checks: nothing overlaps, and the only rows left out are the boundary rows
    assert train.index.max() + pd.Timedelta(hours=h) <= T_END
    assert val.index.min() > train.index.max() and test.index.min() > val.index.max()
    assert val.index.max() + pd.Timedelta(hours=h) < test.index.min()
    assert len(train) + len(val) + len(test) + 2 * h == len(d)

    print(f"\nh = {h}")
    for name, part in splits[h].items():
        tg = part["target"]
        print(f"  {name:<5} rows {len(part):>6}   origins {part.index.min()} -> {part.index.max()}"
              f"   target mean {tg.mean():5.1f}   zeros {(tg == 0).mean():.1%}")

print("\nAll split checks passed")


h = 1
  train rows  12780   origins 2025-01-14 11:00:00 -> 2026-06-30 22:00:00   target mean  30.9   zeros 28.4%
  val   rows   1487   origins 2026-07-01 00:00:00 -> 2026-08-31 22:00:00   target mean  30.1   zeros 28.2%
  test  rows   1391   origins 2026-09-01 00:00:00 -> 2026-10-28 22:00:00   target mean  30.0   zeros 27.7%

h = 3
  train rows  12778   origins 2025-01-14 11:00:00 -> 2026-06-30 20:00:00   target mean  30.9   zeros 28.4%
  val   rows   1485   origins 2026-07-01 00:00:00 -> 2026-08-31 20:00:00   target mean  30.1   zeros 28.1%
  test  rows   1389   origins 2026-09-01 00:00:00 -> 2026-10-28 20:00:00   target mean  30.0   zeros 27.6%

h = 6
  train rows  12775   origins 2025-01-14 11:00:00 -> 2026-06-30 17:00:00   target mean  30.9   zeros 28.4%
  val   rows   1482   origins 2026-07-01 00:00:00 -> 2026-08-31 17:00:00   target mean  30.2   zeros 28.0%
  test  rows   1386   origins 2026-09-01 00:00:00 -> 2026-10-28 17:00:00   target mean  30.1   zeros 27.4%

All split check

In [12]:
def n_events(exceed):
    """Number of separate congestion episodes (runs of consecutive hours above the threshold)."""
    e = pd.Series(exceed.values)
    return int((e & ~e.shift(1, fill_value=False)).sum())

for h in H:
    tr, va, te = splits[h]["train"]["target"], splits[h]["val"]["target"], splits[h]["test"]["target"]
    print(f"\nh = {h}")
    print(f"  {'percentile':<11}{'threshold':>10}{'val hours':>11}{'val events':>12}{'test hours':>12}{'test events':>13}{'test days':>11}")
    for p in [0.80, 0.85, 0.90, 0.95]:
        thr = tr.quantile(p)
        ev_v, ev_t = va >= thr, te >= thr
        days_t = ev_t.groupby(ev_t.index.normalize()).any().sum()
        print(f"  {int(p*100):<11}{thr:>10.1f}{int(ev_v.sum()):>11}{n_events(ev_v):>12}{int(ev_t.sum()):>12}{n_events(ev_t):>13}{int(days_t):>11}")

# Which hours of the day are the alerts in? (test set, 90th percentile, h = 1)
tr1, te1 = splits[1]["train"]["target"], splits[1]["test"]["target"]
thr90 = tr1.quantile(0.90)
hit = te1[te1 >= thr90]
print(f"\nTest alerts at the 90th percentile ({thr90:.1f}), by hour of the target:")
print(hit.groupby((hit.index + pd.Timedelta(hours=1)).hour).size().to_string())
print("\nTest days:", te1.index.normalize().nunique())


h = 1
  percentile  threshold  val hours  val events  test hours  test events  test days
  80               55.0        259          63         222           64         46
  85               60.0        180          50         161           47         46
  90               65.0        126          43         129           45         45
  95               73.0         52          24          77           35         35

h = 3
  percentile  threshold  val hours  val events  test hours  test events  test days
  80               55.0        259          63         222           64         46
  85               60.0        180          50         161           47         46
  90               65.0        126          43         129           45         45
  95               73.0         52          24          77           35         35

h = 6
  percentile  threshold  val hours  val events  test hours  test events  test days
  80               55.0        259          63         222        

In [13]:
from sklearn.preprocessing import StandardScaler

# Columns that are heavily skewed: log(1 + x) transform, using the clipped-at-zero value
skewed = [c for c in X.columns if c.startswith(("jcount", "jlen", "jdelay", "rain_", "precip", "precipitation"))] \
         + ["w_precip"]
skewed = [c for c in skewed if c in clean[1].columns]

def prep(frame):
    f = frame.copy()
    for c in skewed:
        f[c] = np.log1p(f[c].clip(lower=0))
    return f

feature_cols = [c for c in clean[1].columns if c != "target"]
scaled, scalers = {}, {}

for h in H:
    tr, va, te = (prep(splits[h][k]) for k in ["train", "val", "test"])

    sc = StandardScaler().fit(tr[feature_cols])          # fitted on TRAIN only
    scalers[h] = sc

    out = {}
    for name, part in [("train", tr), ("val", va), ("test", te)]:
        z = pd.DataFrame(sc.transform(part[feature_cols]), index=part.index, columns=feature_cols)
        z["target"] = part["target"]                      # target stays in original units
        out[name] = z
    scaled[h] = out

    print(f"\nh = {h}: scaled columns {len(feature_cols)}")
    print("  train mean (should be ~0):", round(out["train"][feature_cols].mean().abs().max(), 4),
          "| train std (should be ~1):", round(out["train"][feature_cols].std().min(), 3), "to", round(out["train"][feature_cols].std().max(), 3))
    print("  val   max abs value:", round(out["val"][feature_cols].abs().max().max(), 1),
          "| test max abs value:", round(out["test"][feature_cols].abs().max().max(), 1))
    print("  missing values:", sum(int(out[k].isna().sum().sum()) for k in out))

print("\nLog-transformed columns:", len(skewed))


h = 1: scaled columns 63
  train mean (should be ~0): 0.0 | train std (should be ~1): 1.0 to 1.0
  val   max abs value: 10.6 | test max abs value: 8.5
  missing values: 0

h = 3: scaled columns 63
  train mean (should be ~0): 0.0 | train std (should be ~1): 1.0 to 1.0
  val   max abs value: 10.6 | test max abs value: 8.5
  missing values: 0

h = 6: scaled columns 63
  train mean (should be ~0): 0.0 | train std (should be ~1): 1.0 to 1.0
  val   max abs value: 10.6 | test max abs value: 8.5
  missing values: 0

Log-transformed columns: 16


In [14]:
for name in ["val", "test"]:
    m = scaled[1][name][feature_cols].abs().max().sort_values(ascending=False)
    print(name, "- largest scaled values:")
    print(m.head(4).round(1).to_string(), "\n")

val - largest scaled values:
w_precip         10.6
precipitation    10.6
precip_lag1      10.6
rain_3h           6.7 

test - largest scaled values:
w_precip         8.5
precipitation    8.5
precip_lag1      8.5
mdelay_lag0      6.5 



In [15]:
L = 24
seq_core    = ["y_lag0", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_sunday", "is_holiday"]
seq_jam     = ["jcount_lag0", "jlen_lag0", "jdelay_lag0", "mdelay_lag0", "tt_lag0"]
seq_weather = groups["weather_basic"]
seq_cols    = seq_core + seq_jam + seq_weather

def make_windows(h, L, seq_cols):
    static_cols = groups_h[h]["calendar_target"] + groups_h[h]["history_target_hour"]

    full = prep(clean[h])
    Z = pd.DataFrame(scalers[h].transform(full[feature_cols]), index=full.index, columns=feature_cols)
    A = Z[seq_cols].to_numpy(dtype="float32")
    S = Z[static_cols].to_numpy(dtype="float32")
    yv = clean[h]["target"].to_numpy(dtype="float32")
    pos = {t: i for i, t in enumerate(Z.index)}

    out = {}
    for name in ["train", "val", "test"]:
        ii = np.array([pos[t] for t in splits[h][name].index])
        ii = ii[ii >= L - 1]                                  # need L hours of history
        Xseq = np.stack([A[i - L + 1: i + 1] for i in ii])
        out[name] = {"seq": Xseq, "static": S[ii], "y": yv[ii], "index": Z.index[ii]}
    return out

windows = {h: make_windows(h, L, seq_cols) for h in H}

for h in H:
    print(f"\nh = {h}")
    for name in ["train", "val", "test"]:
        w = windows[h][name]
        print(f"  {name:<5} seq {w['seq'].shape}  static {w['static'].shape}  y {w['y'].shape}"
              f"   NaN: {int(np.isnan(w['seq']).sum() + np.isnan(w['static']).sum())}")

# Spot check: the last step of a window must equal that origin's scaled row,
# and the first step must be 23 hours earlier
h, t = 3, pd.Timestamp("2026-10-20 19:00")
w = windows[h]["test"]
i = list(w["index"]).index(t)
row = scaled[h]["test"].loc[t]
print("\nSpot check, origin", t, "| h =", h)
print("last step y_lag0 :", round(float(w["seq"][i, -1, 0]), 4), "| scaled row:", round(float(row["y_lag0"]), 4))
print("first step y_lag0:", round(float(w["seq"][i, 0, 0]), 4), "| scaled value 23h earlier:",
      round(float(scaled[h]["test"].loc[t - pd.Timedelta(hours=23), "y_lag0"]), 4))
print("target           :", w["y"][i], "| actual at 22:00:", df.loc[t + pd.Timedelta(hours=h), "TrafficIndexLive"])
print("windows' feature count (sequence):", len(seq_cols), "| static:", w["static"].shape[1])


h = 1
  train seq (12757, 24, 23)  static (12757, 10)  y (12757,)   NaN: 0
  val   seq (1487, 24, 23)  static (1487, 10)  y (1487,)   NaN: 0
  test  seq (1391, 24, 23)  static (1391, 10)  y (1391,)   NaN: 0

h = 3
  train seq (12755, 24, 23)  static (12755, 10)  y (12755,)   NaN: 0
  val   seq (1485, 24, 23)  static (1485, 10)  y (1485,)   NaN: 0
  test  seq (1389, 24, 23)  static (1389, 10)  y (1389,)   NaN: 0

h = 6
  train seq (12752, 24, 23)  static (12752, 10)  y (12752,)   NaN: 0
  val   seq (1482, 24, 23)  static (1482, 10)  y (1482,)   NaN: 0
  test  seq (1386, 24, 23)  static (1386, 10)  y (1386,)   NaN: 0

Spot check, origin 2026-10-20 19:00:00 | h = 3
last step y_lag0 : 0.3817 | scaled row: 0.3817
first step y_lag0: 0.9488 | scaled value 23h earlier: 0.9488
target           : 22.0 | actual at 22:00: 22.0
windows' feature count (sequence): 23 | static: 10


In [16]:
import os, json, joblib

OUT = "../data/processed/model_inputs"
os.makedirs(OUT, exist_ok=True)

# 1. Unscaled tables (for the baselines, SARIMAX, XGBoost and the other tree models)
for h in H:
    for name in ["train", "val", "test"]:
        splits[h][name].to_csv(f"{OUT}/features_h{h}_{name}.csv", index_label="LocalDateTime")

# 2. LSTM windows (one file per horizon)
for h in H:
    arrays = {}
    for name in ["train", "val", "test"]:
        w = windows[h][name]
        arrays[f"{name}_seq"] = w["seq"]
        arrays[f"{name}_static"] = w["static"]
        arrays[f"{name}_y"] = w["y"]
        arrays[f"{name}_index"] = w["index"].to_numpy()
    np.savez_compressed(f"{OUT}/lstm_windows_h{h}.npz", **arrays)

# 3. Scalers (fitted on the training rows only)
for h in H:
    joblib.dump(scalers[h], f"{OUT}/scaler_h{h}.joblib")

# 4. Configuration, so every later notebook uses identical settings
config = {
    "horizons": H,
    "window_length": L,
    "split": {"train_target_end": str(T_END), "val_origin_start": str(V_START),
              "val_target_end": str(V_END), "test_origin_start": str(TEST_START)},
    "alert_threshold_p90_train": float(splits[1]["train"]["target"].quantile(0.90)),
    "alert_threshold_p95_train": float(splits[1]["train"]["target"].quantile(0.95)),
    "sequence_columns": seq_cols,
    "static_columns": groups_h[1]["calendar_target"] + groups_h[1]["history_target_hour"],
    "log_transformed_columns": skewed,
    "feature_groups": {str(h): groups_h[h] for h in H},
    "holidays": [str(d.date()) for d in hol["date"]],
}
with open(f"{OUT}/config.json", "w") as f:
    json.dump(config, f, indent=2)

# 5. Reload and verify
a = pd.read_csv(f"{OUT}/features_h3_test.csv", parse_dates=["LocalDateTime"]).set_index("LocalDateTime")
b = splits[3]["test"]
print("CSV reload matches memory:", a.shape == b.shape and list(a.columns) == list(b.columns)
      and np.allclose(a.to_numpy(), b.to_numpy()))

z = np.load(f"{OUT}/lstm_windows_h3.npz")
print("NPZ reload matches memory:", z["test_seq"].shape == windows[3]["test"]["seq"].shape
      and np.allclose(z["test_seq"], windows[3]["test"]["seq"])
      and (z["test_index"] == windows[3]["test"]["index"].to_numpy()).all())

with open(f"{OUT}/config.json") as f:
    c = json.load(f)
print("Thresholds (p90, p95):", round(c["alert_threshold_p90_train"], 1), round(c["alert_threshold_p95_train"], 1))

print("\nFiles saved:")
for fn in sorted(os.listdir(OUT)):
    print(f"  {fn:<28} {os.path.getsize(os.path.join(OUT, fn)) / 1e6:6.2f} MB")

CSV reload matches memory: True
NPZ reload matches memory: True
Thresholds (p90, p95): 65.0 73.0

Files saved:
  config.json                    0.01 MB
  features_h1_test.csv           0.78 MB
  features_h1_train.csv          6.93 MB
  features_h1_val.csv            0.85 MB
  features_h3_test.csv           0.78 MB
  features_h3_train.csv          6.93 MB
  features_h3_val.csv            0.85 MB
  features_h6_test.csv           0.78 MB
  features_h6_train.csv          6.93 MB
  features_h6_val.csv            0.84 MB
  lstm_windows_h1.npz            1.06 MB
  lstm_windows_h3.npz            1.06 MB
  lstm_windows_h6.npz            1.06 MB
  scaler_h1.joblib               0.00 MB
  scaler_h3.joblib               0.00 MB
  scaler_h6.joblib               0.00 MB
